In [1]:
import pandas as pd
import numpy as np
import joblib
import glob
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
files = sorted(glob.glob("./validation_sets/site_*_val.csv"))
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print("Validation rows loaded:", len(df))

df = df.sort_values(["year","month","day","hour"]).reset_index(drop=True)
df["datetime"] = pd.to_datetime(df[["year","month","day","hour"]])

df["dayofyear"] = df["datetime"].dt.dayofyear
df["weekday"] = df["datetime"].dt.weekday
df["is_weekend"] = df["weekday"].isin([5,6]).astype(int)
df["season"] = (df["month"] % 12) // 3

Validation rows loaded: 25755


In [3]:
lags = [1,2,3,6,12,24]

for lag in lags:
    df[f"O3_lag_{lag}"]  = df["O3_target"].shift(lag)
    df[f"NO2_lag_{lag}"] = df["NO2_target"].shift(lag)

df["O3_roll_6"]  = df["O3_target"].rolling(6).mean()
df["O3_roll_12"] = df["O3_target"].rolling(12).mean()
df["O3_roll_24"] = df["O3_target"].rolling(24).mean()

df["NO2_roll_6"]  = df["NO2_target"].rolling(6).mean()
df["NO2_roll_12"] = df["NO2_target"].rolling(12).mean()
df["NO2_roll_24"] = df["NO2_target"].rolling(24).mean()

df["wind_mag"] = np.sqrt(df["u_forecast"]**2 + df["v_forecast"]**2)
df["temp_NO2"] = df["T_forecast"] * df["NO2_forecast"]
df["wind_NO2"] = df["wind_mag"] * df["NO2_forecast"]

In [4]:
df = df.replace([np.inf,-np.inf], np.nan).dropna().reset_index(drop=True)

features = [
    "O3_forecast","NO2_forecast","T_forecast","q_forecast",
    "u_forecast","v_forecast","w_forecast",
    "NO2_satellite","HCHO_satellite","ratio_satellite",
    "NO2_satellite_missing","HCHO_satellite_missing","ratio_satellite_missing",
    "year","month","day","hour",
    "dayofyear","weekday","is_weekend","season",
    "wind_mag","temp_NO2","wind_NO2",
    "O3_roll_6","O3_roll_12","O3_roll_24",
    "NO2_roll_6","NO2_roll_12","NO2_roll_24"
]
features += [f"O3_lag_{l}" for l in lags]
features += [f"NO2_lag_{l}" for l in lags]

X = df[features]
y_O3 = df["O3_target"]
y_NO2 = df["NO2_target"]

In [5]:
model_O3 = joblib.load("xgb_O3_improved.pkl")
model_NO2 = joblib.load("xgb_NO2_improved.pkl")

pred_O3 = model_O3.predict(X)
pred_NO2 = model_NO2.predict(X)


def evaluate(name, y_true, pred):
    rmse = np.sqrt(mean_squared_error(y_true, pred))
    r2 = r2_score(y_true, pred)
    print(f"\n==== {name} ====")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²  : {r2:.3f}")

evaluate("O3 Validation", y_O3, pred_O3)
evaluate("NO2 Validation", y_NO2, pred_NO2)


==== O3 Validation ====
RMSE: 15.752
R²  : 0.830

==== NO2 Validation ====
RMSE: 16.518
R²  : 0.697


In [6]:
df["O3_pred"] = pred_O3
df["NO2_pred"] = pred_NO2
df.to_csv("validation_predictions_v2.csv", index=False)

print("\n✔ Saved validation_predictions_v2.csv")


✔ Saved validation_predictions_v2.csv
